# Python Data Structure Operations for Signal Sampling Problems

A working reference: run each cell, change the inputs, and see what happens.

**Contents**
1. Sorting dictionaries (by key, by value, ties) and reading them in order
2. NumPy sorting (`sort`, `argsort`, `lexsort`, top-k, `searchsorted`)
3. Other useful structures: lists/tuples, `namedtuple`, `bisect`, `heapq`, `deque`, `Counter`, `defaultdict`, `set`, `itertools`
4. A signal-sampling toolkit that puts them to work (sampling, Nyquist, aliasing, FFT peaks, decimation, interpolation, quantization, moving average)
5. Guided practice exercises (with hints and self-checks)

In [ ]:
import bisect
import heapq
import itertools
from operator import itemgetter
from collections import OrderedDict, Counter, defaultdict, deque, namedtuple

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

---
## 1. Sorting dictionaries

Key facts:
- `sorted(d)` returns a **list of keys**, sorted. It never changes `d`.
- `sorted(d.items())` returns a **list of `(key, value)` tuples**.
- `dict(...)` around that rebuilds a dict in the new order (dicts remember insertion order since Python 3.7).
- `key=` takes a function that says *what to sort by*.

### 1.1 Sort by key

In [ ]:
scores = {"delta": 42, "alpha": 17, "charlie": 99, "bravo": 17, "echo": 63}

print(sorted(scores))                  # keys only, sorted
print(sorted(scores.items()))          # (key, value) pairs sorted by key
by_key = dict(sorted(scores.items()))  # new dict in key order
print(by_key)

# Access in key order WITHOUT building a new dict
for k in sorted(scores):
    print(k, "->", scores[k])

### 1.2 Sort by value

In [ ]:
print(sorted(scores.items(), key=lambda kv: kv[1]))                 # value ascending
print(sorted(scores.items(), key=itemgetter(1)))                    # same, via itemgetter
print(sorted(scores.items(), key=lambda kv: kv[1], reverse=True))   # value descending
print(sorted(scores, key=scores.get))                               # KEYS ordered by their value

by_value = dict(sorted(scores.items(), key=itemgetter(1)))
for k, v in by_value.items():
    print(f"{k:>8} {v}")

### 1.3 Ties and multi-level sorting
`alpha` and `bravo` tie at 17. A tuple key sorts by the first item, then the second, and so on.
Negating a numeric value flips just that level to descending.

In [ ]:
print(sorted(scores.items(), key=lambda kv: (kv[1], kv[0])))     # value asc, then key asc
print(sorted(scores.items(), key=lambda kv: (-kv[1], kv[0])))    # value DESC, then key asc

### 1.4 Extremes without a full sort

In [ ]:
print(max(scores, key=scores.get), min(scores, key=scores.get))
print(heapq.nlargest(3, scores.items(), key=itemgetter(1)))
print(heapq.nsmallest(2, scores.items(), key=itemgetter(1)))

### 1.5 Dict as a `time -> sample` map
Samples that arrive out of order can be read back in time order by sorting the keys.

> Float keys can bite you (`0.1 + 0.2 != 0.3`). For real problems, prefer **integer sample indices** `n` as keys and compute `t = n / fs` when needed.

In [ ]:
samples = {0.02: 0.59, 0.00: 0.0, 0.03: 0.81, 0.01: 0.31}

for t in sorted(samples):
    print(f"t={t:.2f}s  x={samples[t]:+.2f}")

# Turn into two aligned NumPy arrays, in time order
t_arr = np.array(sorted(samples))
x_arr = np.array([samples[t] for t in t_arr])
print(t_arr, x_arr)

# Integer-index version (safer)
fs = 100
by_index = {2: 0.59, 0: 0.0, 3: 0.81, 1: 0.31}
print([(n / fs, by_index[n]) for n in sorted(by_index)])

### 1.6 Insertion order and `OrderedDict`

In [ ]:
d = {"b": 2, "a": 1}
d["c"] = 3
print(list(d))                # insertion order: b, a, c

od = OrderedDict(d)
od.move_to_end("b")           # send to the back
print(list(od))
od.move_to_end("c", last=False)   # send to the front
print(list(od))
print(od.popitem(last=False))     # pop from the front (like a queue)

---
## 2. NumPy sorting

Two ideas to keep separate:
- `np.sort(a)` gives the **sorted values**.
- `np.argsort(a)` gives the **indices** that would sort `a`. This is the one you use when other arrays must stay aligned with `a`.

### 2.1 `sort`, `argsort`, descending

In [ ]:
x = np.array([3.2, -1.5, 8.0, 0.0, -7.3, 4.4])

print(np.sort(x))          # returns a sorted copy
print(np.sort(x)[::-1])    # descending
y = x.copy()
y.sort()                   # in place, returns None
print(y)

idx = np.argsort(x)
print(idx, x[idx])         # x[idx] is the sorted array
print(np.argsort(-x))      # indices for descending order

# rank of each element (0 = smallest)
ranks = np.empty_like(idx)
ranks[idx] = np.arange(len(x))
print(ranks)

### 2.2 Keeping paired arrays aligned (time and value)

In [ ]:
t = np.array([0.3, 0.1, 0.4, 0.0, 0.2])
v = np.array([  5,   1,   8,  -2,   3])

order = np.argsort(t)
t_sorted, v_sorted = t[order], v[order]
print(t_sorted)
print(v_sorted)

# WRONG: sorting them separately destroys the pairing
print(np.sort(t), np.sort(v))

### 2.3 2-D arrays: `axis`, sorting rows by a column, `lexsort`

In [ ]:
A = np.array([[3, 9],
              [1, 7],
              [3, 2],
              [2, 5]])

print(np.sort(A, axis=0))   # each COLUMN sorted independently (rows get scrambled)
print(np.sort(A, axis=1))   # each ROW sorted independently

print(A[np.argsort(A[:, 0], kind="stable")])   # sort ROWS by column 0
print(A[np.lexsort((A[:, 1], A[:, 0]))])       # primary = col 0, tie-break = col 1
# lexsort quirk: the LAST key in the tuple is the primary key

### 2.4 Sorting by magnitude, and top-k

In [ ]:
print(np.argsort(np.abs(x))[::-1])   # indices by |x|, largest first

k = 3
top_k = np.argpartition(x, -k)[-k:]          # k largest, unordered (fast for big arrays)
top_k = top_k[np.argsort(x[top_k])[::-1]]    # now order them, largest first
print(top_k, x[top_k])

### 2.5 Searching sorted data, uniques, and friends

In [ ]:
s = np.sort(x)
print(s)
print(np.searchsorted(s, 3.0))            # where 3.0 would be inserted
print(np.searchsorted(s, [-2, 5]))        # vectorized

vals, counts = np.unique([2, 1, 2, 3, 1, 2], return_counts=True)
print(vals, counts)

print(x.argmax(), x.argmin())             # index of max / min
print(np.where(x > 0)[0], x[x > 0])       # indices / values satisfying a condition

print(np.diff(t_sorted))                  # spacing between samples
print(np.cumsum(v_sorted))                # running sum

# nearest sample to a query time
q = 0.27
print(np.abs(t_sorted - q).argmin(), t_sorted[np.abs(t_sorted - q).argmin()])

---
## 3. Other structures that show up in sampling problems

### 3.1 Lists and tuples of samples

In [ ]:
pts = [(0.2, 3), (0.0, -2), (0.1, 1), (0.1, 0)]

print(sorted(pts))                            # tuples compare element by element
print(sorted(pts, key=lambda p: p[1]))        # by value
print(sorted(pts, key=lambda p: -p[1]))       # by value, descending

pts.sort(key=lambda p: p[0])                  # in place, STABLE: equal keys keep original order
print(pts)

times, vals = zip(*pts)                       # unzip into two tuples
print(times, vals)

### 3.2 `namedtuple`: readable records

In [ ]:
Sample = namedtuple("Sample", ["t", "value"])
ss = [Sample(0.2, 3), Sample(0.0, -2), Sample(0.1, 1)]

print(sorted(ss, key=lambda s: s.t))
print(max(ss, key=lambda s: s.value))
print(ss[0].t, ss[0].value)

### 3.3 `bisect`: fast search in a sorted list

In [ ]:
times = [0.0, 0.1, 0.2, 0.4, 0.5]

i = bisect.bisect_left(times, 0.3)     # insertion point
print(i)
bisect.insort(times, 0.3)              # insert and keep sorted
print(times)


def nearest(sorted_list, q):
    i = bisect.bisect_left(sorted_list, q)
    candidates = sorted_list[max(i - 1, 0): i + 1]
    return min(candidates, key=lambda c: abs(c - q))


print(nearest(times, 0.26), nearest(times, -1), nearest(times, 9))

### 3.4 `heapq`: priority queues, top-k, merging sorted streams

In [ ]:
h = []
for val in [5, 1, 8, 3]:
    heapq.heappush(h, val)
print(heapq.heappop(h), heapq.heappop(h))         # smallest first

# merge two already-sorted streams (e.g. two sensors)
print(list(heapq.merge([0.0, 0.2, 0.4], [0.1, 0.3, 0.5])))

# max-heap trick: push negatives
h = []
for f, m in {5: 1.0, 12: 0.3, 20: 0.5}.items():
    heapq.heappush(h, (-m, f))
print(heapq.heappop(h))                            # (-1.0, 5): strongest component

### 3.5 `deque`: sliding windows

In [ ]:
window = deque(maxlen=3)          # old items fall off automatically
avg = []
for val in [1, 2, 3, 4, 5, 6]:
    window.append(val)
    avg.append(sum(window) / len(window))
print(avg)

dq = deque([1, 2, 3, 4])
dq.rotate(1)                      # circular shift
print(dq)

### 3.6 `Counter` and `defaultdict`: histograms and grouping

In [ ]:
codes = [3, 1, 3, 2, 3, 1, 0]
c = Counter(codes)
print(c)
print(c.most_common(2))
print(sorted(c.items()))          # histogram in level order

# group samples into bins
groups = defaultdict(list)
for val in [0.1, 0.9, 0.4, 0.6, 0.05]:
    groups[int(val * 2)].append(val)   # bins [0,0.5) -> 0, [0.5,1) -> 1
print(dict(groups))

### 3.7 `set`: which frequencies / indices appear where

In [ ]:
a = {5, 12, 20}
b = {12, 20, 33}
print(a | b, a & b, a - b, a ^ b)
print(sorted(a | b))              # sets are unordered: sort before displaying

### 3.8 `itertools`, `zip`, and pairwise operations

In [ ]:
x = [1, 4, 9, 16, 25]
print(list(itertools.accumulate(x)))                 # running sum
print([b - a for a, b in zip(x, x[1:])])             # first difference
print(list(itertools.product([0, 1], repeat=2)))     # all 2-bit codes
print(list(itertools.combinations([1, 2, 3], 2)))

---
## 4. Signal-sampling toolkit

Definitions worked through below:
- Sampling frequency `fs`, sample spacing `Ts = 1/fs`, sample times `t[n] = n / fs`.
- **Nyquist**: a component at `f` is captured faithfully only if `f < fs/2`.
- Above that, it **aliases** to `|f - fs * round(f / fs)|`.

### 4.1 Generate samples

In [ ]:
fs = 100                                   # Hz
N = 100                                    # number of samples -> 1 second
n = np.arange(N)
t = n / fs                                 # same as np.arange(0, 1, 1/fs)
# alternative: np.linspace(0, 1, N, endpoint=False)   (avoid endpoint=True by accident)

x = 1.0 * np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 20 * t)
print(len(x), t[:5], "Nyquist =", fs / 2)

### 4.2 Frequency content and the strongest components

In [ ]:
X = np.fft.rfft(x)
freqs = np.fft.rfftfreq(N, d=1 / fs)
mag = np.abs(X) * 2 / N                    # amplitude (DC and Nyquist bins would not be doubled)

print("peak at", freqs[np.argmax(mag)], "Hz")

top2 = np.argsort(mag)[::-1][:2]           # argsort descending -> top 2 bins
print(freqs[top2], mag[top2])

# dict {frequency: magnitude}, keep only the significant ones, read strongest first
spectrum = {float(f): round(float(m), 3) for f, m in zip(freqs, mag) if m > 0.1}
for f, m in sorted(spectrum.items(), key=itemgetter(1), reverse=True):
    print(f"{f:5.1f} Hz  amplitude {m}")

### 4.3 Aliasing

In [ ]:
def alias(f, fs):
    return abs(f - fs * round(f / fs))


print(alias(30, 40), alias(35, 50), alias(90, 100))

fs2 = 40
f_true = 30
n2 = np.arange(40)
x2 = np.sin(2 * np.pi * f_true * n2 / fs2)

freqs2 = np.fft.rfftfreq(len(x2), d=1 / fs2)
mag2 = np.abs(np.fft.rfft(x2)) * 2 / len(x2)
print("true:", f_true, "Hz | measured peak:", freqs2[np.argmax(mag2)], "Hz")

# see it: the 30 Hz wave and its 10 Hz alias pass through the same samples
tt = np.linspace(0, 0.25, 1000)
plt.figure(figsize=(8, 3))
plt.plot(tt, np.sin(2 * np.pi * 30 * tt), label="30 Hz (true)", alpha=0.6)
plt.plot(tt, -np.sin(2 * np.pi * 10 * tt), "--", label="10 Hz alias (sign flipped)")
plt.stem(n2[:10] / fs2, x2[:10], linefmt="k-", markerfmt="ko", basefmt=" ", label="samples @ 40 Hz")
plt.xlabel("t (s)"); plt.legend(loc="upper right"); plt.tight_layout(); plt.show()

### 4.4 Decimation (downsampling)

In [ ]:
k = 4
x_naive = x[::k]                                    # slicing: fast, but aliasing risk
x_avg = x[: N // k * k].reshape(-1, k).mean(axis=1) # block average: crude low-pass + downsample
print(len(x), "->", len(x_naive), "new fs =", fs // k, "Hz, new Nyquist =", fs // k // 2)

### 4.5 Out-of-order or non-uniform samples -> uniform grid
Steps: sort by time (`argsort`), then `np.interp` (which **requires increasing x-coordinates**).

In [ ]:
rng = np.random.default_rng(1)
t_raw = np.sort(rng.uniform(0, 1, 30))
x_raw = np.sin(2 * np.pi * 3 * t_raw)
perm = rng.permutation(len(t_raw))                  # scramble arrival order
t_in, x_in = t_raw[perm], x_raw[perm]

order = np.argsort(t_in)
t_ok, x_ok = t_in[order], x_in[order]

t_uniform = np.arange(0, 1, 1 / 50)
x_uniform = np.interp(t_uniform, t_ok, x_ok)
print(x_uniform[:5])

### 4.6 Quantization (bits -> levels)

In [ ]:
bits = 3
levels = 2 ** bits
lo, hi = -1.0, 1.0
step = (hi - lo) / levels

sig = np.sin(2 * np.pi * 5 * t)
codes = np.clip(np.floor((sig - lo) / step).astype(int), 0, levels - 1)   # integer level 0..levels-1
recon = lo + (codes + 0.5) * step                                          # mid-point reconstruction

print("step =", step, "| max error =", np.abs(sig - recon).max(), "(<= step/2 =", step / 2, ")")
print(sorted(Counter(codes.tolist()).items()))               # how often each level is used

### 4.7 Moving average, two ways

In [ ]:
w = 5
ma_np = np.convolve(x, np.ones(w) / w, mode="valid")   # vectorized

win, ma_dq = deque(maxlen=w), []                        # streaming
for val in x:
    win.append(val)
    if len(win) == w:
        ma_dq.append(sum(win) / w)

print(np.allclose(ma_np, ma_dq), len(ma_np))

### 4.8 Zero crossings and peaks

In [ ]:
zc = np.where(np.diff(np.signbit(x)))[0]
print("zero crossings at indices", zc[:8], "...")

# simple local maxima: greater than both neighbours
peaks = np.where((x[1:-1] > x[:-2]) & (x[1:-1] > x[2:]))[0] + 1
print("local maxima at", peaks, "| values", x[peaks])

---
## 5. Practice exercises

Each exercise gives you the goal, a few steps, and a check cell. Fill in the `# TODO` line.
The check cells print ✅ or ❌, so re-run them as you go.

In [ ]:
def check(label, ok):
    print(("✅ " if ok else "❌ ") + label)

### Exercise 1: strongest components from a dict

`freq_mag` maps frequency (Hz) to magnitude. Produce `top3`: the **three frequencies with the largest magnitude**, listed in **ascending frequency order**.

Steps:
1. Sort `freq_mag.items()` by the value (index `1` of each pair), descending.
2. Take the first 3 pairs.
3. Keep only the frequencies from those pairs.
4. Sort those frequencies ascending.

Expected for this data: `[5, 20, 47]`

In [ ]:
freq_mag = {5: 1.0, 12: 0.3, 20: 0.5, 33: 0.05, 47: 0.8}

top3 = None   # TODO
print(top3)

In [ ]:
check("Exercise 1", top3 == [5, 20, 47])

### Exercise 2: restore time order

`t_shuf` and `x_shuf` arrived in random order. Produce `t_sorted` and `x_sorted` so the pairs stay matched, then confirm the spacing is uniform.

Steps:
1. Get the ordering indices from `t_shuf` with `np.argsort`.
2. Index **both** arrays with those same indices.
3. Use `np.diff(t_sorted)` and `np.allclose` to test that every gap equals `1/fs`.

In [ ]:
fs = 20
rng = np.random.default_rng(0)
t_true = np.arange(20) / fs
perm = rng.permutation(20)
t_shuf = t_true[perm]
x_shuf = np.sin(2 * np.pi * 3 * t_shuf)

t_sorted = None   # TODO
x_sorted = None   # TODO
uniform = None    # TODO  (True/False)
print(uniform)

In [ ]:
try:
    check("Exercise 2: times sorted", np.allclose(t_sorted, t_true))
    check("Exercise 2: values still paired with times", np.allclose(x_sorted, np.sin(2 * np.pi * 3 * t_true)))
    check("Exercise 2: uniform spacing detected", bool(uniform) is True)
except TypeError:
    print("❌ Exercise 2 not finished yet")

### Exercise 3: predict, then measure, an alias

A 35 Hz tone is sampled at `fs = 50` Hz.

Steps:
1. Predict the apparent frequency with the formula `|f - fs * round(f / fs)|` and store it in `alias_pred`.
2. Build the samples `x3 = sin(2*pi*f*n/fs)` for `n = 0..49` (already given below).
3. Take `np.fft.rfft` and `np.fft.rfftfreq(len(x3), 1/fs)`.
4. Use `np.argmax` on the magnitudes to find the peak frequency and store it in `alias_meas`.

In [ ]:
fs, f = 50, 35
n = np.arange(50)
x3 = np.sin(2 * np.pi * f * n / fs)

alias_pred = None   # TODO
alias_meas = None   # TODO
print(alias_pred, alias_meas)

In [ ]:
try:
    check("Exercise 3: prediction", np.isclose(alias_pred, 15))
    check("Exercise 3: measurement matches prediction", np.isclose(alias_meas, alias_pred))
except TypeError:
    print("❌ Exercise 3 not finished yet")

### Exercise 4: decimate by block averaging

Reduce `x4` by a factor `k = 3` by averaging each block of 3 consecutive samples.

Steps:
1. Check the length is a multiple of `k` (here it is, 12 = 4 x 3).
2. Reshape to `(-1, k)`.
3. Average along the correct axis (each *row* is one block).

Expected: `[1., 4., 7., 10.]`

In [ ]:
x4 = np.arange(12.0)
k = 3

y = None   # TODO
print(y)

In [ ]:
try:
    check("Exercise 4", np.allclose(y, [1, 4, 7, 10]))
except TypeError:
    print("❌ Exercise 4 not finished yet")

---
### Quick cheat sheet

| Goal | Tool |
|---|---|
| Dict keys in order | `sorted(d)` |
| Dict by value | `sorted(d.items(), key=itemgetter(1))` |
| Max / min key by value | `max(d, key=d.get)` |
| Sort values, keep pairing | `order = np.argsort(a); a[order], b[order]` |
| Descending argsort | `np.argsort(-a)` or `np.argsort(a)[::-1]` |
| Sort rows by column j | `A[np.argsort(A[:, j])]` |
| Top-k of a big array | `np.argpartition(a, -k)[-k:]` |
| Insert point / nearest | `np.searchsorted`, `bisect.bisect_left` |
| Sliding window | `deque(maxlen=w)` or `np.convolve` |
| Histogram of levels | `Counter(codes)` |
| Frequency axis | `np.fft.rfftfreq(N, 1/fs)` |
| Alias of `f` | `abs(f - fs * round(f / fs))` |